In [1]:
from datasets import load_dataset

In [2]:
data = load_dataset('jacobmitchinson/colln2003')

In [3]:
data

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [4]:
tags = data['train'].features['ner_tags']
tags.feature.names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

### Define Labels

In [5]:
unique_tags = set()
for doc in data['train']['ner_tags']: 
    for tag in doc:
        unique_tags.add(tag)

unique_tags

{0, 1, 2, 3, 4, 5, 6, 7, 8}

In [6]:
id2label = {i: f"TAG_{i}" for i in sorted(unique_tags)}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

In [7]:
id2label , label2id , num_labels

({0: 'TAG_0',
  1: 'TAG_1',
  2: 'TAG_2',
  3: 'TAG_3',
  4: 'TAG_4',
  5: 'TAG_5',
  6: 'TAG_6',
  7: 'TAG_7',
  8: 'TAG_8'},
 {'TAG_0': 0,
  'TAG_1': 1,
  'TAG_2': 2,
  'TAG_3': 3,
  'TAG_4': 4,
  'TAG_5': 5,
  'TAG_6': 6,
  'TAG_7': 7,
  'TAG_8': 8},
 9)

### Load the Tokenizer

In [8]:
from transformers import AutoTokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

### Tokenize the Dataset

In [19]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation = True,
        is_split_into_words = True ,
        padding = False
    )
    
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index = i)
        aligned_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(label[word_idx])
            else:
                # for subwords inside the same word
                aligned_labels.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx
        labels.append(aligned_labels)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


label_all_tokens = True

tokenized_dataset = data.map(tokenize_and_align_labels, batched = True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

### Load DistilBERT for Token Classification

In [20]:
from transformers import DistilBertForTokenClassification

In [21]:
model = DistilBertForTokenClassification.from_pretrained(
   model_checkpoint,
   num_labels=num_labels,
   id2label = id2label,
   label2id = label2id
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Define Training Arguments

In [22]:
from transformers import TrainingArguments , DataCollatorForTokenClassification

In [23]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [13]:
training_args = TrainingArguments(
    output_dir = "./distilbert-ner",
    eval_strategy = "epoch",
    learning_rate = 5e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 2,
    weight_decay = 0.01,
    save_strategy = "epoch",
    logging_steps = 50,
    load_best_model_at_end = True,
    metric_for_best_model = "f1"
)

In [14]:
from seqeval.metrics import accuracy_score, f1_score, classification_report

In [15]:
def compute_metrics(p):
    predictions , labels = p 
    predictions = predictions.argmax(axis = -1)
    
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return { 
       "accuracy": accuracy_score(true_labels, true_predictions),
       "f1": f1_score(true_labels, true_predictions)
    }

### Initializa Trainer

In [16]:
from transformers import Trainer

In [24]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["validation"],
    tokenizer = tokenizer,
    compute_metrics = compute_metrics,
    data_collator=data_collator
)

C:\Users\tipto\AppData\Local\Temp\ipykernel_8392\2474204414.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.074000,0.060855,0.982175,0.938739
2,0.021100,0.057453,0.984908,0.948405


c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_5 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_7 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_8 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\

TrainOutput(global_step=1756, training_loss=0.08106567536253728, metrics={'train_runtime': 62.5821, 'train_samples_per_second': 448.722, 'train_steps_per_second': 28.059, 'total_flos': 340476837529122.0, 'train_loss': 0.08106567536253728, 'epoch': 2.0})

In [26]:
results = trainer.evaluate()

c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_5 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_7 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: TAG_8 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\tipto\

In [27]:
results

{'eval_loss': 0.05745340511202812,
 'eval_accuracy': 0.9849079384243887,
 'eval_f1': 0.9484046594074448,
 'eval_runtime': 2.375,
 'eval_samples_per_second': 1368.435,
 'eval_steps_per_second': 85.896,
 'epoch': 2.0}